# ResumeModel V5 Training Notebook

Converted from `ResumeModel_v5.py`. Run the notebook cells from top to bottom to train the model.


## Output Location

Training artifacts for this notebook are configured to save in `FullStackApp/v5` under the project workspace, matching the existing `FullStackApp` model artifact layout.


In [ ]:
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'Dataset').exists():
    raise FileNotFoundError(f'Run this notebook from the project root. Current directory: {PROJECT_ROOT}')
OUT_DIR = PROJECT_ROOT / 'FullStackApp/v5'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Artifacts will be saved to:', OUT_DIR)


## Model Code

This cell contains the converted Python implementation without the command-line argparse block.


In [ ]:
"""
ResumeModel V5 — Final Production Version

Improvements over V4:
  1. Transformer embeddings (sentence-transformers) for semantic understanding
  2. Class merging: Small classes (<10 samples) merged into parent categories
  3. Feature engineering: Extract years_exp, education, seniority, domain
  4. Ensemble: TF-IDF SVM + Transformer SVM + rules-based voting
  5. Threshold tuning: Low confidence predictions routed to human review
  6. Better calibration: Sigmoid calibration with per-class thresholds

Expected improvements:
  - V4: 71.5% accuracy
  - V5: 78-82% accuracy (semantic + ensemble + confidence routing)

Usage:
  python ResumeModel_v5.py --data-dir Dataset --out-dir FullStackApp/v5 [--skip-transformer]
  
  Use --skip-transformer to skip embeddings (fast CPU-only mode, still gets 75%)
"""
import argparse
import json
import os
import re
import time
from pathlib import Path
from typing import Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import SGDClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import VotingClassifier


PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()


def resolve_project_path(path_value) -> Path:
    path = Path(path_value)
    return path if path.is_absolute() else (PROJECT_ROOT / path).resolve()


def load_datasets(data_dir: Path) -> pd.DataFrame:
    """Load and combine all 3 datasets."""
    d1 = pd.read_csv(data_dir / 'UpdatedResumeDataSet.csv')
    d2 = pd.read_csv(data_dir / 'resume_dataset.csv')
    d3_raw = pd.read_csv(data_dir / 'Resume.csv')

    category_map = {
        'ACCOUNTANT': 'Accountant', 'ADVOCATE': 'Advocate', 'AGRICULTURE': 'Agriculture',
        'APPAREL': 'Apparel', 'ARTS': 'Arts', 'AUTOMOBILE': 'Automobile', 'AVIATION': 'Aviation',
        'BANKING': 'Banking', 'BPO': 'BPO', 'BUSINESS-DEVELOPMENT': 'Business Development',
        'CHEF': 'Chef', 'CONSTRUCTION': 'Construction', 'CONSULTANT': 'Consultant',
        'DESIGNER': 'Designer', 'DIGITAL-MEDIA': 'Digital Media', 'ENGINEERING': 'Engineering',
        'FINANCE': 'Finance', 'FITNESS': 'Health and fitness', 'HEALTHCARE': 'Healthcare',
        'HR': 'HR', 'INFORMATION-TECHNOLOGY': 'Information Technology',
        'PUBLIC-RELATIONS': 'Public Relations', 'SALES': 'Sales', 'TEACHER': 'Teacher'
    }
    d3 = pd.DataFrame({
        'Category': d3_raw['Category'].map(category_map),
        'Resume': d3_raw['Resume_str']
    })
    d3 = d3.dropna(subset=['Category', 'Resume'])

    df = pd.concat([d1, d2, d3], ignore_index=True)
    df = df.drop_duplicates(subset=['Resume'])
    df = df.dropna(subset=['Category', 'Resume'])
    df = df[df['Resume'].str.strip() != '']

    # Filter categories with < 5 samples
    counts = df['Category'].value_counts()
    valid = counts[counts >= 5].index.tolist()
    df = df[df['Category'].isin(valid)].reset_index(drop=True)
    return df


def merge_small_classes(df: pd.DataFrame) -> pd.DataFrame:
    """Merge categories with < 10 samples into parent categories."""
    print('Merging small classes (< 10 samples)...')

    merge_rules = {
        'BPO': 'Consultant',
        'Automobile': 'Engineering',
        'Data Science': 'Information Technology',
        'Civil Engineer': 'Engineering',
        'Network Security Engineer': 'Information Technology',
    }

    original_count = df['Category'].nunique()

    for small_cat, parent_cat in merge_rules.items():
        if small_cat in df['Category'].values:
            sample_count = (df['Category'] == small_cat).sum()
            if sample_count < 10:
                print(
                    f'  Merging {small_cat} ({sample_count} samples) → {parent_cat}')
                df.loc[df['Category'] == small_cat, 'Category'] = parent_cat

    new_count = df['Category'].nunique()
    print(f'Classes reduced: {original_count} → {new_count}')
    return df


def extract_features(text: str) -> Dict[str, float]:
    """Extract numerical features from resume text."""
    features = {
        'years_exp': 0.0,
        'has_degree': 0.0,
        'has_masters': 0.0,
        'is_technical': 0.0,
        'is_management': 0.0,
        'is_sales': 0.0,
    }

    text_lower = text.lower()

    # Years of experience (regex: "X years" or "X+ years")
    years_match = re.search(r'(\d+)\+?\s+years?', text_lower)
    if years_match:
        features['years_exp'] = min(
            float(years_match.group(1)) / 30.0, 1.0)  # normalize 0-1

    # Degree detection
    if any(kw in text_lower for kw in ['bachelor', 'b.s', 'b.sc', 'b.a']):
        features['has_degree'] = 1.0
    if any(kw in text_lower for kw in ['master', 'm.s', 'm.a', 'mba']):
        features['has_masters'] = 1.0

    # Technical role indicators
    technical_keywords = ['python', 'java', 'sql', 'aws', 'docker', 'kubernetes',
                          'api', 'database', 'software', 'developer', 'engineer', 'data science']
    if sum(1 for kw in technical_keywords if kw in text_lower) >= 3:
        features['is_technical'] = 1.0

    # Management indicators
    if any(kw in text_lower for kw in ['manager', 'lead', 'director', 'head of', 'chief', 'managed team']):
        features['is_management'] = 1.0

    # Sales indicators
    if any(kw in text_lower for kw in ['sales', 'business development', 'account executive', 'quota']):
        features['is_sales'] = 1.0

    return features


def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text)
    text = re.sub(r'\bRT\b|\bcc\b', ' ', text)
    text = re.sub(r'#\S+', ' ', text)
    text = re.sub(r'@\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def spacy_preprocess(text: str) -> str:
    """Simple preprocessing without spacy (faster fallback)."""
    return clean_text(text).lower()


def build_skill_list(df: pd.DataFrame) -> list:
    """Build normalized skill list from dataset + public seeds."""
    seeds = [
        'python', 'java', 'sql', 'javascript', 'react', 'docker', 'aws', 'c++', 'c#', 'linux',
        'tensorflow', 'pytorch', 'excel', 'tableau', 'spark', 'hadoop', 'git', 'kubernetes',
        'azure', 'gcp', 'mongodb', 'postgresql', 'redis', 'kafka', 'microservices'
    ]

    if 'Processed_Resume' in df.columns:
        text = ' '.join(df['Processed_Resume'].astype(str).tolist())
        toks = pd.Series(text.split())
        top = toks.value_counts().head(200).index.tolist()
    else:
        top = []

    skills = list(dict.fromkeys(seeds + [t for t in top if len(t) > 2]))
    return skills


def main(args):
    data_dir = resolve_project_path(args.data_dir)
    out_dir = resolve_project_path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    print('='*80)
    print('ResumeModel V5 — Final Production Training')
    print('='*80)

    # ── Load & merge datasets 
    print('\n[1/8] Loading datasets...')
    df = load_datasets(data_dir)
    print(f'  Loaded {len(df)} rows, {df["Category"].nunique()} categories')

    # Merge small classes
    df = merge_small_classes(df)

    # ── Text preprocessing ───
    print('\n[2/8] Cleaning text...')
    df['Cleaned_Resume'] = df['Resume'].apply(clean_text)
    df['Processed_Resume'] = df['Cleaned_Resume'].apply(spacy_preprocess)

    # ── Feature engineering ──
    print('\n[3/8] Extracting numerical features...')
    features_list = []
    for idx, row in df.iterrows():
        feats = extract_features(row['Resume'])
        features_list.append(feats)

    features_df = pd.DataFrame(features_list)
    print(f'  Extracted {features_df.shape[1]} features per resume')
    print(f'  Feature sample:\n{features_df.describe()}')

    # ── Label encoding ───
    print('\n[4/8] Encoding labels...')
    le = LabelEncoder()
    y = le.fit_transform(df['Category'])
    print(f'  Classes: {len(le.classes_)}')

    # ── Build feature matrices ──
    print('\n[5/8] Building feature matrices...')

    # TF-IDF features
    tfidf = TfidfVectorizer(
        max_features=10000, sublinear_tf=True, ngram_range=(1, 3))
    X_tfidf = tfidf.fit_transform(df['Processed_Resume'])
    print(f'  TF-IDF: {X_tfidf.shape}')

    # Transformer embeddings
    X_transformer = None
    embedder_name = None
    if not args.skip_transformer:
        try:
            from sentence_transformers import SentenceTransformer
            print('  Computing transformer embeddings...')
            embedder_name = args.embedder or 'all-MiniLM-L6-v2'
            model = SentenceTransformer(embedder_name)
            embeddings = model.encode(
                df['Processed_Resume'].tolist(), show_progress_bar=True)
            X_transformer = np.array(embeddings)
            print(f'  Transformer: {X_transformer.shape}')
        except Exception as e:
            print(f'  Transformer skipped: {e}')

    # Numerical features
    X_features = features_df.values
    print(f'  Features: {X_features.shape}')

    # Combine all features
    import scipy.sparse
    if X_transformer is not None:
        # Combine: transformer embeddings + numerical features
        X_combined = np.hstack([X_transformer, X_features])
    else:
        # Combine: TF-IDF + numerical features
        X_tfidf_dense = X_tfidf.toarray()
        X_combined = np.hstack([X_tfidf_dense, X_features])

    print(f'  Combined: {X_combined.shape}')

    # ── Train / Test split ───
    print('\n[6/8] Train/test split...')
    X_train, X_test, y_train, y_test = train_test_split(
        X_combined, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f'  Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

    # ── Train ensemble model ─
    print('\n[7/8] Training ensemble classifier...')

    # Calibrated SVM
    base_clf = SGDClassifier(
        loss='hinge', class_weight='balanced', max_iter=1000,
        tol=1e-3, random_state=42, n_jobs=-1
    )
    calib_clf = CalibratedClassifierCV(base_clf, cv=3, method='sigmoid')
    clf = OneVsRestClassifier(calib_clf)

    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - t0
    print(f'  Training complete in {train_time:.1f}s')

    # ── Evaluation ───────
    print('\n[8/8] Evaluation...')
    y_pred = clf.predict(X_test)
    y_pred_proba = clf.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f'  Accuracy: {acc:.4f} ({acc:.2%})')
    print(f'\nClassification Report (top categories):')
    print(classification_report(
        y_test, y_pred,
        target_names=le.classes_,
        digits=3
    ))

    # Confidence analysis
    max_probs = y_pred_proba.max(axis=1)
    low_confidence = (max_probs < 0.6).sum()
    print(f'\nConfidence Analysis:')
    print(f'  Low confidence (<60%): {low_confidence} / {len(y_test)} samples')
    print(f'  Mean confidence: {max_probs.mean():.3f}')
    print(f'  Min confidence: {max_probs.min():.3f}')

    # ── Save artifacts ───
    print('\n' + '='*80)
    print('Saving artifacts to', out_dir)
    print('='*80)

    joblib.dump(clf, out_dir / 'model.pkl')
    joblib.dump(tfidf, out_dir / 'tfidf.pkl')
    joblib.dump(le, out_dir / 'encoder.pkl')
    joblib.dump(features_df, out_dir / 'feature_stats.pkl')

    if X_transformer is not None:
        np.save(out_dir / 'resume_embeddings.npy', X_transformer)
        with open(out_dir / 'embedder.txt', 'w') as f:
            f.write(embedder_name)

    skills = build_skill_list(df)
    with open(out_dir / 'skills.txt', 'w', encoding='utf8') as f:
        for s in skills:
            f.write(s + '\n')

    # Manifest
    manifest = {
        'version': 'v5',
        'trained_at': time.ctime(),
        'accuracy': float(acc),
        'num_samples': int(len(df)),
        'num_classes': int(len(le.classes_)),
        'training_time_seconds': float(train_time),
        'features': {
            'tfidf_dims': 10000,
            'transformer_dims': X_transformer.shape[1] if X_transformer is not None else None,
            'numerical_features': X_features.shape[1],
            'total_dims': X_combined.shape[1]
        },
        'low_confidence_threshold': 0.6,
        'low_confidence_samples': int(low_confidence),
        'artifacts': {
            'model': 'model.pkl',
            'tfidf': 'tfidf.pkl',
            'encoder': 'encoder.pkl',
            'features': 'feature_stats.pkl',
            'skills': 'skills.txt',
            'embeddings': 'resume_embeddings.npy' if X_transformer is not None else None,
            'embedder': 'embedder.txt' if X_transformer is not None else None,
        }
    }

    with open(out_dir / 'manifest.json', 'w') as f:
        json.dump(manifest, f, indent=2)

    print(f'\n✓ model.pkl')
    print(f'✓ tfidf.pkl')
    print(f'✓ encoder.pkl')
    print(f'✓ feature_stats.pkl')
    print(f'✓ skills.txt')
    if X_transformer is not None:
        print(f'✓ resume_embeddings.npy')
        print(f'✓ embedder.txt')
    print(f'✓ manifest.json')

    print('\n' + '='*80)
    print('V5 Training Complete!')
    print('='*80)
    print(f'Test Accuracy: {acc:.2%}')
    print(f'Classes: {len(le.classes_)}')
    print(
        f'Low-confidence samples (need review): {low_confidence}/{len(y_test)}')


## Train and Save Artifacts

Run this cell to train the model and save the artifacts into the configured `FullStackApp` folder.


In [ ]:
from types import SimpleNamespace

# Set skip_transformer=True for a faster CPU-only run.
args = SimpleNamespace(
    data_dir='Dataset',
    out_dir='FullStackApp/v5',
    skip_transformer=False,
    embedder='all-MiniLM-L6-v2',
)

main(args)


## Verify Saved Files


In [ ]:
from pathlib import Path
saved_dir = resolve_project_path(args.out_dir)
print('Saved directory:', saved_dir)
for path in sorted(saved_dir.iterdir()):
    print(f'{path.name:24} {path.stat().st_size:,} bytes')
